We will ensure that our identified best models perform better than baselin models (random -- TF labels permuted, linear regression) on test data (as assessed by EMD loss). Once confirmed, we will re-train a single model using the "best paramaters" on the training + validation data and assess it's performance on the test data. This is simply for the sake of simplicity, to avoid having 5 models that we are assessing.

In [1]:
import os
import scanpy as sc
from tqdm import trange

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from geomloss import SamplesLoss

import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler

import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.insert(1, '../.')
from Kang_utils import get_best_hyperparams


[KeOps] Warning : There were warnings or errors :
<stdin>:1:10: fatal error: cuda.h: No such file or directory
compilation terminated.

[KeOps] Warning : 
    The location of Cuda header files cuda.h and nvrtc.h could not be detected on your system.
    You must determine their location and then define the environment variable CUDA_PATH,
    either before launching Python or using os.environ before importing keops. For example
    if these files are in /vol/cuda/10.2.89-cudnn7.6.4.38/include you can do :
      import os
      os.environ['CUDA_PATH'] = '/vol/cuda/10.2.89-cudnn7.6.4.38'
    
[KeOps] Compiling cuda jit compiler engine ... 
[KeOps] Warning : There were warnings or errors :
/nobackup/users/hmbaghda/Software/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/keopscore/binders/nvrtc/nvrtc_jit.cpp:16:10: fatal error: cuda.h: No such file or directory
 #include <cuda.h>
          ^~~~~~~~
compilation terminated.

OK
[pyKeOps] Compiling nvrtc binder for python ... 
[KeOps] W

In [2]:
import sys
sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import io
from scLEMBAS.model.scl import SignalingModel
from scLEMBAS.model.train import TrainSC
from scLEMBAS import io
from scLEMBAS.model.model_components import FCLayers
from scLEMBAS.model.lr_schedulers import WarmupCosineAnnealingWarmRestarts


In [3]:
seed = 888
device = "cuda" if torch.cuda.is_available() else "cpu"

data_path = '/nobackup/users/hmbaghda/scLEMBAS/analysis'
models_path = os.path.join(data_path, 'processed', 'models')

In [4]:
adata = sc.read_h5ad(os.path.join(data_path, 'processed', 'kang_expr_scored.h5ad'))
sn_ppis = pd.read_csv(os.path.join(data_path, 'processed', 'Kang_sn_ppis.csv'), index_col = 0)

tf_adata = io.read_tfad(os.path.join(data_path, 'processed', 'Kang_tf_activity.h5ad'))
tf_adata.obs['condition'] = tf_adata.obs['stim'].astype(str) + '^' + tf_adata.obs['seurat_annotations'].astype(str)

/nobackup/users/hmbaghda/Software/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
source_label = 'source_genesymbol'
target_label = 'target_genesymbol'
weight_label = 'mode_of_action'
stimulation_label = 'consensus_stimulation'
inhibition_label = 'consensus_inhibition'

Load best models: 

In [6]:
res = pd.read_csv(os.path.join(data_path, 'processed', 'Kang_k_fold_validation_results.csv'), index_col = 0)
best_emd_mean, best_hyperparams, best_emd = get_best_hyperparams(res)

trainers_best = io.read_pickled_object(os.path.join(data_path, 'processed', 'Kang_trainers_k.pickle'))

In [7]:
# sanity check
for k, trainer_k in trainers_best.items():
    trainer_params = [trainer_k.hyper_params['max_epochs'], trainer_k.hyper_params['maximum_learning_rate'], 
             trainer_k.hyper_params['train_batch_size'], trainer_k.hyper_params['vae_scaling_KL']]
    recorded_params = [best_hyperparams['max_epochs'], best_hyperparams['max_lr'], best_hyperparams['train_batch_size'], 
best_emd[best_emd.k == k]['KL_regularization'].tolist()[0]]
    if trainer_params != recorded_params:
        raise ValueError('The stored models do not match the best hyper params')

        


On each fold, let's train a random model with the same parameters and compare the predictions on the test data with the actual model. We also calculate the loss of the random output vs the actual output, which is equivalent to if the random model had perfect predictions on the random data.

# Start

In [8]:
rand_emd_k, actual_emd_k, untrained_rand_emd_k = [], [], []
stim_map = {'STIM': 1, 'CTRL': 0}

test_cells = open(os.path.join(data_path, 'processed', 'data_split_barcodes', 'kang_test.txt')).read().splitlines()


In [9]:
k = 0
trainer_k = trainers_best[k]

In [10]:
mod_actual = trainer_k.mod

train_cells = trainer_k.X_train.index.tolist()

In [13]:
# ------------train the linear model------------
y_out = mod_actual.y_out.copy()
X_in = mod_actual.X_in.copy()
cats = mod_actual.signaling_network.covariates.copy()
expr = mod_actual.expr.copy()

expr_train = expr.loc[train_cells, :]
cat_dummies = pd.get_dummies(cats, drop_first=True).astype(int)
scaler = MinMaxScaler(feature_range=(0, 1))
expr_train_scaled = scaler.fit_transform(expr_train)
expr_train_scaled = pd.DataFrame(expr_train_scaled, columns = expr_train.columns, index = expr_train.index)
X_train_linear = pd.concat([X_in.loc[train_cells, :], cat_dummies.loc[train_cells, :], expr_train_scaled], 
         ignore_index = False, axis = 1)

# mod_linear.fit(X_train_linear, y_out.loc[train_cells, :])

now, instead of a linear regression, provide code for a simple FFN from pytorch

provide some code in pytorch to take a multi sample multifeature input X_in (mxn) where m are samples and n is features to a multi sample multi feature output y_out (mxp) where m are samples and p is features

In [20]:
def train_FFN(X_train_FFN, y_train_FFN, trainer_k):
    ffn_train_loader = DataLoader(TensorDataset(X_train_FFN, y_train_ffn), 
                                  batch_size=trainer_k.hyper_params['train_batch_size'], 
                                  shuffle=True)

    n_layers_ffn = 3
    ffn_layers = list(np.round(np.linspace(X_train_FFN.shape[1], y_out.shape[1], n_layers_ffn + 2)).astype(int))
    mod_ffn = FCLayers(layers = ffn_layers, dtype = mod_actual.dtype, device = device)
    ffn_optimizer = torch.optim.Adam(mod_ffn.parameters(), 
                                             lr=trainer_k.hyper_params['maximum_learning_rate'], 
                                             weight_decay=trainer_k.hyper_params['bn_weights_lambda_l2'])
    reset_state = ffn_optimizer.state.copy()
    lr_scheduler = WarmupCosineAnnealingWarmRestarts(optimizer = ffn_optimizer,
                                                          T_0 = trainer_k.hyper_params['lr_restart_epoch'],
                                                          T_mul = trainer_k.hyper_params['lr_restart_factor'], 
                                                          gamma = trainer_k.hyper_params['lr_decay'],
                                                          eta_min = trainer_k.hyper_params['minimum_learning_rate'],
                                                          max_lr=trainer_k.hyper_params['maximum_learning_rate'],
                                                          warmup_steps = trainer_k.hyper_params['warmup_epochs'],
                                                          last_epoch = -1)
    ffn_loss_fn = SamplesLoss("sinkhorn", p=2, blur=0.05)#.to(device)

    for e in trange(2):
        for X_batch, y_batch in ffn_train_loader:
            mod_ffn.train()
            ffn_optimizer.zero_grad()

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            y_pred_ffn = mod_ffn(X_batch)
            ffn_loss = ffn_loss_fn(y_pred_ffn, y_batch)

            ffn_loss.backward()
            ffn_optimizer.step()
        lr_scheduler.step()
        if np.logical_and(e % trainer_k.hyper_params['reset_optimizer_epoch'] == 0, e>0):
            ffn_optimizer.state = reset_state.copy()
    return mod_ffn

In [ ]:
X_train_FFN = pd.concat([X_in.loc[train_cells, :], cat_dummies.loc[train_cells, :], expr_train.copy()], 
     ignore_index = False, axis = 1)
X_train_FFN = torch.tensor(X_train_FFN.values, dtype = mod_actual.dtype, device = device)
y_train_ffn = torch.tensor(y_out.loc[train_cells, :].values, dtype = mod_actual.dtype, device = device)
mod_ffn = train_FFN(X_train_FFN, y_train_ffn, trainer_k)

In [23]:
for cond in tf_adata.obs.loc[test_cells, :].condition.unique():
    stim, ct = cond.split('^')

    test_cells_cond = tf_adata.obs[(tf_adata.obs.index.isin(test_cells)) & (tf_adata.obs['condition'] == cond)].index.tolist()
    y_test = mod_actual.df_to_tensor(tf_adata.to_df().loc[test_cells_cond, :])

    # in distribution gene expression!
    expr_test = mod_actual.df_to_tensor(mod_actual.expr.loc[train_cells, :])

    # set the stimulation condition we want to predict
    X_test_df = pd.DataFrame(data = {'IFNB1': [stim_map[stim]]*len(train_cells)})
    X_test = mod_actual.df_to_tensor(X_test_df)

    # set the cell type we want to predict
    cov_idx_map = dict(zip(mod_actual.signaling_network.covariates['seurat_annotations'], 
                           mod_actual.signaling_network.covariates_idx['seurat_annotations']))
    covariates_idx_test = torch.tensor([cov_idx_map[ct]]*len(train_cells), device = mod_actual.device, dtype = torch.int64).view(-1,1)

    break

In [24]:
# linear model
cat_dummies_test = cat_dummies.loc[train_cells, :].reset_index(drop = True)
cat_dummies_test[:] = 0
cat_dummies_test['_'.join(['seurat_annotations', ct])] = 1
X_test_linear = pd.concat([X_test_df, cat_dummies_test, expr_train_scaled.reset_index(drop = True)],
                          axis = 1)
# y_pred_linear = mod_linear.predict(X_test_linear)
# y_pred_linear = torch.tensor(y_pred_linear, dtype = mod_actual.dtype, device = mod_actual.device)
# loss_fn = SamplesLoss("sinkhorn", p=2, blur=0.05).to(device)
# tot_linear += loss_fn(y_pred_linear, y_test).detach().cpu().item()


In [31]:
X_test_linear

,IFNB1,seurat_annotations_B Activated,seurat_annotations_CD4 Memory T,seurat_annotations_CD4 Naive T,seurat_annotations_CD8 T,seurat_annotations_CD14 Mono,seurat_annotations_CD16 Mono,seurat_annotations_DC,seurat_annotations_Eryth,seurat_annotations_Mk,...,ZNF285,ZNF471,XXbac-B444P24.14,SERHL2,AF127936.7,AP000476.1,AF131217.1,TMPRSS3,AP001626.1,AP001062.7
0,1,0,0,0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,0,0,0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,0,0,0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,0,0,0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,0,0,0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10009,1,0,0,0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10010,1,0,0,0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10011,1,0,0,0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10012,1,0,0,0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,IFNB1,seurat_annotations_B Activated,seurat_annotations_CD4 Memory T,seurat_annotations_CD4 Naive T,seurat_annotations_CD8 T,seurat_annotations_CD14 Mono,seurat_annotations_CD16 Mono,seurat_annotations_DC,seurat_annotations_Eryth,seurat_annotations_Mk,...,ZNF285,ZNF471,XXbac-B444P24.14,SERHL2,AF127936.7,AP000476.1,AF131217.1,TMPRSS3,AP001626.1,AP001062.7
0,1,0,0,0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,0,0,0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,0,0,0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,0,0,0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,0,0,0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10009,1,0,0,0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10010,1,0,0,0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10011,1,0,0,0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10012,1,0,0,0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [40]:
loss_fn = SamplesLoss("sinkhorn", p=2, blur=0.05).to(device)
# tot_linear += loss_fn(y_pred_linear, y_test).detach().cpu().item()

tensor(130.0444, device='cuda:0', grad_fn=<SelectBackward0>)

# End

100%|█████████████████████████████████████████████| 2/2 [00:01<00:00,  1.39it/s]


In [137]:
rand_emd_k, actual_emd_k, untrained_rand_emd_k, linear_emd_k, ffn_emd_k = [], [], [], [], []
stim_map = {'STIM': 1, 'CTRL': 0}

test_cells = open(os.path.join(data_path, 'processed', 'data_split_barcodes', 'kang_test.txt')).read().splitlines()

for k, trainer_k in trainers_best.items():
    mod_actual = trainer_k.mod
    
    train_cells = trainer_k.X_train.index.tolist()
    
    # ------------permute the tf labels for random model------------
    y_out = mod_actual.y_out.copy()
    np.random.seed(seed)
    permuted_tfs = np.random.permutation(y_out.columns)
    rand_y_out = y_out[permuted_tfs].copy()
    
    # ------------train the linear model------------
    X_in = mod_actual.X_in.copy()
    cats = mod_actual.signaling_network.covariates.copy()
    expr = mod_actual.expr.copy()

    expr_train = expr.loc[train_cells, :]
    cat_dummies = pd.get_dummies(cats, drop_first=True).astype(int)
    scaler = MinMaxScaler(feature_range=(0, 1))
    expr_train_scaled = scaler.fit_transform(expr_train)
    expr_train_scaled = pd.DataFrame(expr_train_scaled, columns = expr_train.columns, index = expr_train.index)
    X_train_linear = pd.concat([X_in.loc[train_cells, :], cat_dummies.loc[train_cells, :], expr_train_scaled], 
             ignore_index = False, axis = 1)
    mod_linear = LinearRegression()
    mod_linear.fit(X_train_linear, y_out.loc[train_cells, :])
    
    # ------------train the FFN------------
    X_train_FFN = pd.concat([X_in.loc[train_cells, :], cat_dummies.loc[train_cells, :], expr_train.copy()], 
         ignore_index = False, axis = 1)
    X_train_FFN = torch.tensor(X_train_FFN.values, dtype = mod_actual.dtype, device = device)
    y_train_ffn = torch.tensor(y_out.loc[train_cells, :].values, dtype = mod_actual.dtype, device = device)
    mod_ffn = train_FFN(X_train_FFN, y_train_ffn, trainer_k)

    # ------------build and train the random model------------
    mod_rand = SignalingModel(net = sn_ppis,
                         X_in = mod_actual.X_in.copy(),
                         y_out = rand_y_out, 
                         expr = mod_actual.expr.copy(), 
                         covariates = mod_actual.signaling_network.covariates.copy(),
                         categorical_covariate_keys = mod_actual.signaling_network.covariates.columns.tolist(),
                         projection_amplitude_in = mod_actual.input_layer.projection_amplitude, 
                         projection_amplitude_out = mod_actual.projection_amplitude_out,
                         weight_label = weight_label, source_label = source_label, target_label = target_label,
                         bionet_params = mod_actual.signaling_network.bionet_params.copy(), 
                         dtype = torch.float32, device = device, seed = seed)
    mod_rand.input_layer.weights.requires_grad = False # don't learn scaling factors for the ligand input concentrations
    mod_rand.signaling_network.prescale_weights(target_radius = mod_actual.signaling_network.bionet_params['spectral_target']) # spectral radius
    
    hyper_params = trainer_k.hyper_params
    hyper_params['validation_batch_size'] = np.nan
    hyper_params['test_batch_size'] = len(test_cells)
    trainer = TrainSC(mod = mod_rand,
                      prediction_optimizer = torch.optim.Adam,
                      prediction_loss_fn = SamplesLoss("sinkhorn", p=2, blur=0.05).to(device), #torch.nn.MSELoss(reduction='mean'),
                      discriminator_params = trainer_k.discriminator['params'].copy(),
                      hyper_params = hyper_params.copy(),
                  train_split = {'train': train_cells, 'test': test_cells, 'validation': None},
                  train_seed = seed,
                  track_test = False,
                  track_validation = False)

    mod_rand = trainer.train_model(verbose = False)

    # ---------------------------------evaluation---------------------------------
    # calculate loss on test for each of the actual and random model
    # for each condition: get the test condition predictions from all in-distribution gene expression that was trained
    # take the total loss across conditions
    tot_linear, tot_rand, tot_rand_untrained, tot_actual, tot_ffn = 0, 0, 0, 0, 0
    for cond in tf_adata.obs.loc[test_cells, :].condition.unique():
        stim, ct = cond.split('^')
        
        test_cells_cond = tf_adata.obs[(tf_adata.obs.index.isin(test_cells)) & (tf_adata.obs['condition'] == cond)].index.tolist()
        y_test = mod_actual.df_to_tensor(tf_adata.to_df().loc[test_cells_cond, :])
        
        # in distribution gene expression!
        expr_test = mod_actual.df_to_tensor(mod_actual.expr.loc[train_cells, :])
        
        # set the stimulation condition we want to predict
        X_test_df = pd.DataFrame(data = {'IFNB1': [stim_map[stim]]*len(train_cells)})
        X_test = mod_actual.df_to_tensor(X_test_df)

        # set the cell type we want to predict
        cov_idx_map = dict(zip(mod_actual.signaling_network.covariates['seurat_annotations'], 
                               mod_actual.signaling_network.covariates_idx['seurat_annotations']))
        covariates_idx_test = torch.tensor([cov_idx_map[ct]]*len(train_cells), device = mod_actual.device, dtype = torch.int64).view(-1,1)

        # ------------linear model------------
        cat_dummies_test = cat_dummies.loc[train_cells, :].reset_index(drop = True)
        cat_dummies_test[:] = 0
        cat_dummies_test['_'.join(['seurat_annotations', ct])] = 1
        X_test_linear = pd.concat([X_test_df, cat_dummies_test, expr_train_scaled.reset_index(drop = True)],
                                  axis = 1)
        y_pred_linear = mod_linear.predict(X_test_linear)
        y_pred_linear = torch.tensor(y_pred_linear, dtype = mod_actual.dtype, device = mod_actual.device)
        loss_fn = SamplesLoss("sinkhorn", p=2, blur=0.05).to(device)
        tot_linear += loss_fn(y_pred_linear, y_test).detach().cpu().item()
        
        # ------------FFN------------
        X_test_FFN = torch.tensor(pd.concat([X_test_df, cat_dummies_test, expr_train.reset_index(drop = True)], axis = 1).values, 
                                  device = device, dtype = mod_actual.dtype)
        y_pred_FFN = mod_ffn(X_test_FFN)
        loss_fn = SamplesLoss("sinkhorn", p=2, blur=0.05).to(device)
        tot_ffn += loss_fn(y_pred_FFN, y_test).detach().cpu().item()
        
        # ------------random prediction - perfect------------
        y_rand = mod_actual.df_to_tensor(tf_adata.to_df().loc[test_cells_cond, rand_y_out.columns])
        loss_fn = SamplesLoss("sinkhorn", p=2, blur=0.05).to(device)
        tot_rand_untrained += loss_fn(y_rand, y_test).detach().cpu().item()

        # ------------random model - trained------------
        mod_rand.eval()
        with torch.inference_mode():
            y_predicted_rand, _, _ = mod_rand(X_in = X_test, covariates_idx = covariates_idx_test, expr = expr_test)
        loss_fn = SamplesLoss("sinkhorn", p=2, blur=0.05).to(device)
        tot_rand += loss_fn(y_predicted_rand, y_test).detach().cpu().item()

        # ------------actual model------------
        mod_actual.eval()
        with torch.inference_mode():
            y_predicted_actual, _, _ = mod_actual(X_in = X_test, covariates_idx = covariates_idx_test, expr = expr_test)
        loss_fn = SamplesLoss("sinkhorn", p=2, blur=0.05).to(device)
        tot_actual += loss_fn(y_predicted_actual, y_test).detach().cpu().item()

        del expr_test, X_test, covariates_idx_test, y_predicted_rand, y_predicted_actual
        del loss_fn, y_rand, X_test_df, X_test_FFN, y_pred_FFN
        torch.cuda.empty_cache()
        
    rand_emd_k.append(tot_rand)
    actual_emd_k.append(tot_actual)
    untrained_rand_emd_k.append(tot_rand_untrained)
    linear_emd_k.append(tot_linear)
    ffn_emd_k.append(tot_ffn)

    k_fold_test = pd.DataFrame(data = {'actual': actual_emd_k, 
                                      'rand': rand_emd_k, 
                                      'perfectly_rand': untrained_rand_emd_k, 
                                      'linear': linear_emd_k, 
                                      'FFN': ffn_emd_k})
    k_fold_test.to_csv(os.path.join(data_path, 'processed', 'k_fold_random.csv'))
    

Visualize results:

In [42]:
# fig, ax = plt.subplots(figsize = (5,5))

# pval = mannwhitneyu(k_fold_test.actual, k_fold_test.random, alternative = 'less').pvalue

# viz_df = pd.melt(k_fold_test, var_name='model', value_name='Total EMD Loss')

# sns.violinplot(data = viz_df, x = 'model', y = 'Total EMD Loss', ax = ax)
# ax.annotate('MWU p-val: {:3f}'.format(pval),
#             xy=(0.5, 0.9),
#             xycoords='axes fraction')